# Current manuscript Figure 3 — exact-layout replication

This notebook is intentionally **not a redesign**. Its first goal is to
reproduce the structure of the current manuscript Figure 3 using the cleaned
`002_decode_msaa` outputs.

Current manuscript layout:

- **A:** full-reconstruction decoding across K — spatial AA
- **B:** full-reconstruction decoding across K — temporal AA
- **C:** intact-only top-m decoding across K — spatial AA
- **D:** intact-only top-m decoding across K — temporal AA

The visual conventions match the manuscript:
- condition colors: intact = purple, word = green, rest = black
- top row uses shaded 95% confidence intervals
- bottom row overlays top-m subsets
- representative labels: K=48 (spatial) and K=14 (temporal)

This notebook is the clean replication layer. New PCA / held-out analyses
should be added as separate figures or explicit updated variants rather than
silently changing the original figure's design.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SPATIAL_DIR = Path("msaa_condrank_decoding_outputs_spatial_across")
TEMPORAL_DIR = Path("msaa_condrank_decoding_outputs_temporal_across")

SPATIAL_FULL = SPATIAL_DIR / "full_summary.csv"
TEMPORAL_FULL = TEMPORAL_DIR / "full_summary.csv"

SPATIAL_TOPM = SPATIAL_DIR / "topm_summary.csv"
TEMPORAL_TOPM = TEMPORAL_DIR / "topm_summary.csv"

for p in [SPATIAL_FULL, TEMPORAL_FULL, SPATIAL_TOPM, TEMPORAL_TOPM]:
    if not p.exists():
        raise FileNotFoundError(
            f"Missing required file: {p}\n"
            "Run 002_decode_msaa.ipynb for both spatial/across and "
            "temporal/across first."
        )

spatial_full = pd.read_csv(SPATIAL_FULL)
temporal_full = pd.read_csv(TEMPORAL_FULL)
spatial_topm = pd.read_csv(SPATIAL_TOPM)
temporal_topm = pd.read_csv(TEMPORAL_TOPM)

CONDITIONS = ["intact", "word", "rest"]

COND_COLORS = {
    "intact": "purple",
    "word": "green",
    "rest": "black",
}

COND_LABELS = {
    "intact": "Intact",
    "word": "Word",
    "rest": "Rest",
}

PREFERRED_TOP_M = [1, 2, 3, 5, 7, 10]

TOPM_COLORS = {
    1: "#1f77b4",
    2: "#ff7f0e",
    3: "#2ca02c",
    5: "#d62728",
    7: "#9467bd",
    10: "#8c564b",
}

REPRESENTATIVE_K = {
    "spatial": 48,
    "temporal": 14,
}

OUTPUT_DIR = Path("paper_figure_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FIG_PNG = OUTPUT_DIR / "Figure3_replication.png"
FIG_PDF = OUTPUT_DIR / "Figure3_replication.pdf"

DPI = 300

print("Spatial K:", sorted(spatial_full["K"].astype(int).unique()))
print("Temporal K:", sorted(temporal_full["K"].astype(int).unique()))
print("Spatial top-m:", sorted(spatial_topm["top_m"].astype(int).unique()))
print("Temporal top-m:", sorted(temporal_topm["top_m"].astype(int).unique()))

In [ ]:
def current_rows(df, analysis_type):
    out = df.copy()

    if "analysis_type" in out.columns:
        out = out[out["analysis_type"].astype(str) == analysis_type]

    if "fit_scope" in out.columns:
        out = out[out["fit_scope"].astype(str) == "across"]

    if "K" in out.columns:
        out["K"] = out["K"].astype(int)

    if "top_m" in out.columns:
        out["top_m"] = out["top_m"].astype(int)

    return out


spatial_full = current_rows(spatial_full, "spatial")
temporal_full = current_rows(temporal_full, "temporal")
spatial_topm = current_rows(spatial_topm, "spatial")
temporal_topm = current_rows(temporal_topm, "temporal")


def ci95_from_sem(df):
    # 002_decode_msaa stores err as SEM across decoding fold/repetition rows.
    # The manuscript caption describes 95% confidence intervals, so the
    # plotting layer converts SEM to approximately 95% CI as 1.96 * SEM.
    if "err" not in df.columns:
        return None
    return 1.96 * df["err"].astype(float).to_numpy()


def plot_full_panel(ax, df, title):
    for condition in CONDITIONS:
        cur = (
            df[df["condition"].astype(str) == condition]
            .sort_values("K")
        )

        if len(cur) == 0:
            continue

        x = cur["K"].to_numpy(dtype=float)
        y = cur["mean"].to_numpy(dtype=float)
        ci = ci95_from_sem(cur)

        ax.plot(
            x,
            y,
            color=COND_COLORS[condition],
            marker="o",
            markersize=2.8,
            linewidth=1.6,
            label=COND_LABELS[condition],
        )

        if ci is not None:
            ax.fill_between(
                x,
                y - ci,
                y + ci,
                color=COND_COLORS[condition],
                alpha=0.12,
                linewidth=0,
            )

    ax.set_title(title, fontsize=12)
    ax.set_xlabel("Number of archetypes (K)", fontsize=10)
    ax.set_ylabel("Across-condition decoding accuracy", fontsize=10)
    ax.tick_params(labelsize=9)
    ax.spines[["top", "right"]].set_visible(False)


def plot_topm_panel(ax, df, title, representative_k):
    intact = df[df["condition"].astype(str) == "intact"].copy()

    available = sorted(intact["top_m"].dropna().astype(int).unique())
    top_ms = [m for m in PREFERRED_TOP_M if m in available]

    if 7 not in available:
        print(
            f"{title or 'top-m panel'}: top-7 is not present in the cleaned output. "
            f"Plotting available values {top_ms}."
        )

    for top_m in top_ms:
        cur = intact[intact["top_m"] == top_m].sort_values("K")

        x = cur["K"].to_numpy(dtype=float)
        y = cur["mean"].to_numpy(dtype=float)
        ci = ci95_from_sem(cur)

        ax.errorbar(
            x,
            y,
            yerr=ci,
            color=TOPM_COLORS.get(top_m),
            marker="o",
            markersize=2.7,
            linewidth=1.35,
            capsize=2,
            elinewidth=0.8,
            label=f"top-{top_m}",
        )

    ax.set_title(title, fontsize=12)
    ax.set_xlabel("Number of archetypes (K)", fontsize=10)
    ax.set_ylabel(
        "Top-m decoding accuracy for intact condition",
        fontsize=10,
    )
    ax.tick_params(labelsize=9)
    ax.spines[["top", "right"]].set_visible(False)

    ymax = ax.get_ylim()[1]
    ax.text(
        representative_k,
        ymax * 0.97,
        f"K={representative_k}",
        ha="left",
        va="top",
        fontsize=9,
    )

    return top_ms

In [ ]:
fig, axes = plt.subplots(
    2,
    2,
    figsize=(10.2, 7.6),
    gridspec_kw={
        "hspace": 0.34,
        "wspace": 0.28,
    },
)

axA, axB = axes[0]
axC, axD = axes[1]

plot_full_panel(
    axA,
    spatial_full,
    "Spatial AA",
)

plot_full_panel(
    axB,
    temporal_full,
    "Temporal AA",
)

handles, labels = axB.get_legend_handles_labels()
axB.legend(
    handles,
    labels,
    title="Condition",
    frameon=False,
    fontsize=8.5,
    title_fontsize=9,
    loc="center right",
)

spatial_topms = plot_topm_panel(
    axC,
    spatial_topm,
    "",
    representative_k=REPRESENTATIVE_K["spatial"],
)

temporal_topms = plot_topm_panel(
    axD,
    temporal_topm,
    "",
    representative_k=REPRESENTATIVE_K["temporal"],
)

handles, labels = axD.get_legend_handles_labels()
axD.legend(
    handles,
    labels,
    title="Top-m",
    frameon=False,
    fontsize=8.3,
    title_fontsize=9,
    loc="upper right",
)

for ax, letter in zip(
    [axA, axB, axC, axD],
    ["A", "B", "C", "D"],
):
    ax.text(
        -0.08,
        1.06,
        letter,
        transform=ax.transAxes,
        fontsize=13,
        fontweight="normal",
        ha="left",
        va="bottom",
        clip_on=False,
    )

if len(spatial_full):
    axA.set_xlim(
        spatial_full["K"].min() - 5,
        spatial_full["K"].max() + 15,
    )
if len(temporal_full):
    axB.set_xlim(
        temporal_full["K"].min() - 3,
        temporal_full["K"].max() + 8,
    )
if len(spatial_topm):
    axC.set_xlim(
        spatial_topm["K"].min() - 5,
        spatial_topm["K"].max() + 15,
    )
if len(temporal_topm):
    axD.set_xlim(
        temporal_topm["K"].min() - 3,
        temporal_topm["K"].max() + 8,
    )

fig.subplots_adjust(
    left=0.11,
    right=0.97,
    top=0.96,
    bottom=0.10,
)

fig.savefig(
    FIG_PNG,
    dpi=DPI,
    bbox_inches="tight",
)

fig.savefig(
    FIG_PDF,
    bbox_inches="tight",
)

plt.show()

print("Saved:", FIG_PNG)
print("Saved:", FIG_PDF)

## What this notebook is for

This notebook is the **clean replication** of the current manuscript Figure 3.

It is intentionally separate from:
- the new empirical PCA benchmark,
- the participant-held-out top-m sensitivity,
- the simulation benchmark,
- the selected-archetype interpretation figure.

Those new reviewer-driven analyses should be added as new figures or explicit
updated variants after the original manuscript figures have been reproduced
cleanly.